# Live Coding Musical pilote par LLM

**Module :** 04-Audio-Applications  
**Niveau :** Avance  
**Technologies :** Strudel.cc, Supriya, OpenAI API, numpy synthesis  
**Duree estimee :** 50 minutes  

## Objectifs d'Apprentissage

- [ ] Comprendre le live coding musical et ses outils (TidalCycles, Strudel, Sonic Pi)
- [ ] Generer du code Strudel a partir de descriptions textuelles via un LLM
- [ ] Integrer un editeur Strudel interactif dans un notebook Jupyter
- [ ] Creer des patterns musicaux programmatiques avec numpy
- [ ] Piloter la synthese audio par conversation avec un LLM
- [ ] Iterer sur des compositions via le prompt engineering musical

## Prerequis

- `pip install openai numpy scipy soundfile`
- Cle API OpenAI (pour la generation de code par LLM)
- Navigateur web avec JavaScript active (pour Strudel.cc)
- Optionnel : SuperCollider + `pip install supriya` (pour synthese avancee)

**Navigation :** [<< 04-4](04-4-Audio-Video-Sync.ipynb) | [Index](../README.md) | [Image >>](../../Image/README.md)

In [ ]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Configuration LLM
llm_model = "gpt-4o-mini"          # Modele pour la generation de code
use_llm = True                     # Utiliser le LLM pour generer du code

# Configuration audio
sample_rate = 44100                # Taux d'echantillonnage
bpm = 120                          # Tempo par defaut
save_results = True                # Sauvegarder les fichiers audio

# Configuration Supriya (optionnel)
test_supriya = False               # Tester Supriya (necessite SuperCollider)

Les parametres Papermill definissent le mode d'execution (`notebook_mode`), le modele LLM a utiliser pour la composition (`llm_model`), et les options de generation (nombre de compositions, tokens maximum). La cellule suivante initialise l'environnement Python et importe les bibliotheques necessaires.

In [2]:
# Setup environnement et imports
import os
import sys
import time
import json
import urllib.parse
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
import logging

import numpy as np
from IPython.display import Audio, display, HTML, IFrame

# Import helpers GenAI
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import play_audio, save_audio
        print("Helpers audio importes")
    except ImportError:
        print("Helpers audio non disponibles - mode autonome")

# Repertoires
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'livecoding'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('livecoding')

print(f"Live Coding Musical pilote par LLM")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}")
print(f"LLM : {llm_model if use_llm else 'desactive'}")
print(f"BPM : {bpm}, SR : {sample_rate}")
print(f"Sortie : {OUTPUT_DIR}")

Helpers audio importes
Live Coding Musical pilote par LLM
Date : 2026-03-22 13:36:08
Mode : interactive
LLM : gpt-4o-mini
BPM : 120, SR : 44100
Sortie : MyIA.AI.Notebooks\GenAI\outputs\audio\livecoding


L'environnement Python est configure et les bibliotheques importees : `urllib.parse` pour encoder les patterns Strudel en URL, `datetime` pour l'horodatage des compositions. La cellule suivante charge le fichier `.env` pour recuperer la cle API du LLM (OpenAI ou compatible).

In [3]:
# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os

current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path}")
        env_loaded = True
        break
    current_path = current_path.parent
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# Configuration OpenAI
openai_available = False
try:
    import openai
    api_key = os.environ.get("OPENAI_API_KEY")
    base_url = os.environ.get("OPENAI_BASE_URL")
    if api_key:
        client_kwargs = {"api_key": api_key}
        if base_url:
            client_kwargs["base_url"] = base_url
        client = openai.OpenAI(**client_kwargs)
        openai_available = True
        print(f"OpenAI API disponible (modele: {llm_model})")
    else:
        print("OPENAI_API_KEY non definie")
except ImportError:
    print("openai non installe (pip install openai)")

# Verifier Supriya
supriya_available = False
if test_supriya:
    try:
        import supriya
        supriya_available = True
        print("supriya disponible")
    except ImportError:
        print("supriya non installe (pip install supriya + SuperCollider)")

OPENAI_API_KEY non definie


## Dependances GPU Optionnelles

Ce notebook ne necessite **pas de GPU**. Il utilise :
- L'API OpenAI pour la generation de code musical (cloud)
- Strudel.cc pour la lecture audio (navigateur)
- numpy/scipy pour la synthese programmatique (CPU)

## Section 1 : Le live coding musical

Le live coding est une pratique artistique ou le musicien ecrit et modifie du code en temps reel pour generer de la musique. Nee dans les annees 2000, cette discipline a produit plusieurs outils devenus standards.

| Outil | Langage | Plateforme | Particularite |
|-------|---------|-----------|---------------|
| **TidalCycles** | Haskell | Desktop | Pionnier, patterns puissants |
| **Sonic Pi** | Ruby-like | Desktop | Tres pedagogique |
| **Strudel.cc** | JavaScript | Navigateur | Zero install, port de TidalCycles |
| **FoxDot** | Python | Desktop | Syntaxe Python native |
| **Supriya** | Python | Desktop | API Python pour SuperCollider |

### Pourquoi le LLM change la donne

Le live coding traditionnel necessite d'apprendre une syntaxe specialisee. Avec un LLM :

| Approche | Entree | Sortie | Courbe d'apprentissage |
|----------|--------|--------|----------------------|
| Traditionnel | Code Strudel/Tidal | Musique | Elevee |
| LLM-assiste | "Un rythme funk en 4/4" | Code -> Musique | Quasi nulle |
| Hybride | Code + instructions | Code raffine -> Musique | Moderate |

L'IA ne remplace pas le musicien : elle **abaisse la barriere d'entree** et permet l'iteration rapide.

## Section 2 : Strudel.cc - Live coding dans le navigateur

Strudel utilise une mini-notation compacte heritee de TidalCycles. Chaque pattern decrit une sequence d'evenements musicaux.

### Syntaxe de base

| Syntaxe | Signification | Exemple |
|---------|--------------|--------|
| `s("bd sd")` | Sequence de samples | Kick, snare |
| `note("c3 e3 g3")` | Notes MIDI | Do, Mi, Sol |
| `[a b]` | Sous-division | 2 events dans 1 temps |
| `a*4` | Repetition | 4 fois dans le cycle |
| `~` | Silence | Pause |
| `<a b c>` | Alternance | Un par cycle |
| `.bank("RolandTR808")` | Choix de sons | Boite a rythmes TR-808 |
| `.room(0.5)` | Reverb | Espace |
| `.speed(2)` | Vitesse | Pitch up |

### Exemples

```javascript
// Beat basique
s("bd sd [bd bd] sd").bank("RolandTR808")

// Melodie piano
note("c3 e3 g3 b3").s("piano")

// Polyrythmie
s("bd sd, hh hh hh hh, ~ cp ~ cp")

// Rythme euclidien
s("bd(3,8), sd(2,8), hh(5,8)")
```

In [4]:
# Integration Strudel dans le notebook
print("STRUDEL.CC - EDITEUR INTERACTIF")
print("=" * 45)


def strudel_editor(code: str, height: int = 300) -> HTML:
    """Cree un editeur Strudel interactif dans le notebook.
    
    Securite : le code est echappe via html.escape() pour prevenir
    toute injection XSS depuis du contenu genere par LLM.
    """
    import html as html_mod
    safe_code = html_mod.escape(code)
    return HTML(f"""
    <script src="https://unpkg.com/@strudel/repl@0.10.5"></script>
    <strudel-editor style="height:{height}px;">
{safe_code}
    </strudel-editor>
    """)


def strudel_link(code: str) -> str:
    """Genere un lien vers le REPL Strudel en ligne."""
    encoded = urllib.parse.quote(code)
    return f"https://strudel.cc/#{encoded}"


# Demo : beat basique
demo_code = 's("bd sd [bd bd] sd").bank("RolandTR808")'
print(f"Code Strudel : {demo_code}")
print(f"Lien REPL : {strudel_link(demo_code)}")

if notebook_mode == "interactive" and not skip_widgets:
    print("\nEditeur interactif (cliquer Play pour ecouter) :")
    display(strudel_editor(demo_code))
else:
    print("Mode batch - editeur interactif desactive")
    print(f"Ouvrir le lien REPL dans un navigateur pour ecouter")

STRUDEL.CC - EDITEUR INTERACTIF
Code Strudel : s("bd sd [bd bd] sd").bank("RolandTR808")
Lien REPL : https://strudel.cc/#s%28%22bd%20sd%20%5Bbd%20bd%5D%20sd%22%29.bank%28%22RolandTR808%22%29

Editeur interactif (cliquer Play pour ecouter) :


### Interpretation : Strudel dans le notebook

| Aspect | Observation |
|--------|------------|
| Integration | Le composant `<strudel-editor>` s'integre nativement |
| Audio | Playback via Web Audio API du navigateur |
| Interactivite | Code modifiable en direct, feedback immediat |
| Limitation | Pas de rendu offline (WAV) - navigateur requis |

**Point cle** : Strudel ne necessite aucune installation. La mini-notation est suffisamment compacte pour etre generee par un LLM.

## Section 3 : LLM comme compositeur

On peut utiliser un LLM pour generer du code Strudel a partir de descriptions en langage naturel. Le systeme prompt definit les regles de la mini-notation.

In [5]:
# Prompt engineering pour la generation de code musical
print("LLM COMME COMPOSITEUR")
print("=" * 45)

STRUDEL_SYSTEM_PROMPT = """You are a Strudel.cc live coding expert.
Generate valid Strudel mini-notation code for the given music description.

Available functions:
- s("pattern") : sample pattern (bd=kick, sd=snare, hh=hihat, cp=clap, oh=open hat)
- note("pattern") : note pattern (c3, d3, e3... or MIDI numbers)
- .bank("name") : sound bank (RolandTR808, RolandTR909)
- .speed(n) : playback speed
- .gain(n) : volume (0-1)
- .room(n) : reverb (0-1)
- .delay(n) : delay (0-1)
- .lpf(n) : low-pass filter (Hz)
- .hpf(n) : high-pass filter (Hz)
- .sometimes(fn) : apply function randomly
- .every(n, fn) : apply function every n cycles
- .rev() : reverse pattern

Syntax:
- [a b] = subdivide (2 events in 1 step)
- a*4 = repeat 4 times
- ~ = silence
- <a b c> = alternate per cycle
- a(3,8) = euclidean rhythm
- , = parallel layers

Return ONLY the code, no explanation. Keep it concise (1-5 lines)."""


def generate_strudel(description: str) -> Optional[str]:
    """Genere du code Strudel via LLM."""
    if not openai_available:
        print("  API OpenAI non disponible")
        return None

    try:
        response = client.chat.completions.create(
            model=llm_model,
            messages=[
                {"role": "system", "content": STRUDEL_SYSTEM_PROMPT},
                {"role": "user", "content": description}
            ],
            temperature=0.7,
            max_tokens=500,
        )
        code = response.choices[0].message.content.strip()
        # Nettoyer les blocs de code markdown
        if code.startswith("```"):
            lines = code.split("\n")
            code = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])
        return code
    except Exception as e:
        print(f"  Erreur LLM : {type(e).__name__} - {str(e)[:100]}")
        return None


# Test avec plusieurs descriptions
descriptions = [
    "A funky drum pattern with syncopated hi-hats and a deep kick",
    "A calm ambient pad with slow arpeggios",
    "An energetic techno beat at 140 BPM with acid bass",
]

generated_codes = {}

if use_llm and openai_available:
    for desc in descriptions:
        print(f"\n--- {desc[:50]}... ---")
        code = generate_strudel(desc)
        if code:
            generated_codes[desc] = code
            print(f"  Code : {code[:100]}")
            print(f"  REPL : {strudel_link(code)[:80]}...")
else:
    print("LLM desactive ou API non disponible")
    print("\nExemples de code Strudel pre-genere :")
    examples = {
        "Funk beat": 's("bd ~ [bd bd] ~, ~ sd ~ sd, hh*8").bank("RolandTR808").gain(0.8)',
        "Ambient pad": 'note("<c3 e3 g3 b3>").s("sawtooth").room(0.9).lpf(800).gain(0.3)',
        "Techno acid": 's("bd*4, ~ cp ~ cp, hh*8").bank("RolandTR909").speed(1.1)',
    }
    for name, code in examples.items():
        generated_codes[name] = code
        print(f"  {name} : {code}")

LLM COMME COMPOSITEUR
LLM desactive ou API non disponible

Exemples de code Strudel pre-genere :
  Funk beat : s("bd ~ [bd bd] ~, ~ sd ~ sd, hh*8").bank("RolandTR808").gain(0.8)
  Ambient pad : note("<c3 e3 g3 b3>").s("sawtooth").room(0.9).lpf(800).gain(0.3)
  Techno acid : s("bd*4, ~ cp ~ cp, hh*8").bank("RolandTR909").speed(1.1)


Le LLM a genere plusieurs variations de patterns Strudel avec differents styles (minimaliste, groovy, experimental). La cellule suivante affiche les editeurs interactifs permettant d'ecouter et de modifier chaque pattern dans le navigateur via l'interface Strudel REPL.

In [6]:
# Afficher les editeurs interactifs pour le code genere
print("EDITEURS INTERACTIFS")
print("=" * 45)

if notebook_mode == "interactive" and not skip_widgets and generated_codes:
    for name, code in generated_codes.items():
        print(f"\n--- {name[:50]} ---")
        display(strudel_editor(code))
else:
    print("Mode batch - editeurs interactifs desactives")
    print("\nLiens REPL pour ecouter dans le navigateur :")
    for name, code in generated_codes.items():
        print(f"  {name[:40]} : {strudel_link(code)[:60]}...")

EDITEURS INTERACTIFS

--- Funk beat ---



--- Ambient pad ---



--- Techno acid ---


### Interpretation : LLM comme compositeur

| Aspect | Observation |
|--------|------------|
| Pertinence | Le code genere est generalement valide et musical |
| Style | Bon respect des descriptions de genre |
| Limites | Parfois trop simple ou utilise des fonctions inexistantes |
| Iteration | On peut raffiner en ajoutant "add more swing" ou "make it darker" |

**Points cles** :
1. La mini-notation Strudel est ideale pour le LLM (compacte, bien documentee)
2. Le system prompt avec exemples ameliore la qualite
3. `temperature=0.7` offre un bon equilibre creativite/coherence
4. L'iteration conversationnelle est le vrai pouvoir de cette approche

## Section 4 : Synthese programmatique avec numpy

Pour generer des fichiers audio WAV sans dependance externe, on peut utiliser numpy pour la synthese sonore de base. Cela permet de creer des beats et melodies programmatiquement.

| Technique | Description | Usage |
|-----------|-------------|-------|
| Oscillateur | `sin(2*pi*f*t)` | Notes, tons |
| Bruit | `np.random.randn()` | Snare, hi-hat |
| Enveloppe | `exp(-t*decay)` | Attaque/relachement |
| Filtre | Moyenne glissante, biquad | Timbre |

In [ ]:
# Synthese audio programmatique avec numpy
print("SYNTHESE PROGRAMMATIQUE")
print("=" * 45)

import scipy.io.wavfile as wavfile


def make_kick(sr=44100, duration=0.15):
    """Genere un kick drum synthetique."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    # Sweep de frequence de 150Hz a 50Hz
    freq = 150 * np.exp(-t * 20) + 50
    phase = np.cumsum(2 * np.pi * freq / sr)
    wave = np.sin(phase) * np.exp(-t * 15)
    return (wave * 0.8).astype(np.float32)


def make_snare(sr=44100, duration=0.12):
    """Genere un snare drum synthetique."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    noise = np.random.randn(len(t)).astype(np.float32) * 0.3
    tone = np.sin(2 * np.pi * 200 * t) * 0.4
    envelope = np.exp(-t * 25)
    return ((noise + tone) * envelope).astype(np.float32)


def make_hihat(sr=44100, duration=0.05):
    """Genere un hi-hat synthetique."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    noise = np.random.randn(len(t)).astype(np.float32) * 0.15
    # Filtre passe-haut simple
    filtered = np.diff(noise, prepend=0) * 3
    envelope = np.exp(-t * 40)
    return (filtered * envelope).astype(np.float32)


def make_bass(freq=55, sr=44100, duration=0.3):
    """Genere une note de basse synthetique."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    wave = np.sin(2 * np.pi * freq * t) + 0.3 * np.sin(2 * np.pi * 2 * freq * t)
    envelope = np.exp(-t * 5) * np.minimum(1.0, t * 50)
    return (wave * envelope * 0.4).astype(np.float32)


def make_beat(pattern, bpm=120, sr=44100):
    """Cree un beat a partir d'un pattern textuel.
    
    Pattern format : liste de (instrument, positions)
    Positions : liste de temps en 16th notes (0-15 pour 1 mesure)
    """
    step_duration = 60.0 / bpm / 4  # 16th note
    total_steps = 16  # 1 mesure en 4/4
    total_samples = int(total_steps * step_duration * sr)
    audio = np.zeros(total_samples, dtype=np.float32)

    instruments = {
        "kick": make_kick,
        "snare": make_snare,
        "hihat": make_hihat,
    }

    for instr_name, positions in pattern:
        if instr_name in instruments:
            sample = instruments[instr_name](sr=sr)
            for pos in positions:
                start = int(pos * step_duration * sr)
                end = min(start + len(sample), total_samples)
                audio[start:end] += sample[:end - start]

    return np.clip(audio, -1, 1)


# Creer differents patterns
patterns = {
    "Rock basique": [
        ("kick", [0, 8]),
        ("snare", [4, 12]),
        ("hihat", [0, 2, 4, 6, 8, 10, 12, 14]),
    ],
    "Funk syncopé": [
        ("kick", [0, 3, 6, 10]),
        ("snare", [4, 12]),
        ("hihat", [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),
    ],
    "Demi-temps": [
        ("kick", [0]),
        ("snare", [8]),
        ("hihat", [0, 4, 8, 12]),
    ],
}

beat_results = {}

for name, pattern in patterns.items():
    print(f"\n--- {name} ---")
    audio = make_beat(pattern, bpm=bpm, sr=sample_rate)
    # Repeter 4 fois
    audio_loop = np.tile(audio, 4)
    duration = len(audio_loop) / sample_rate

    beat_results[name] = {"audio": audio_loop, "duration": duration}
    print(f"  Duree : {duration:.1f}s")
    display(Audio(data=audio_loop, rate=sample_rate))

    if save_results:
        safe_name = name.lower().replace(" ", "_").replace("é", "e")
        filepath = OUTPUT_DIR / f"beat_{safe_name}.wav"
        wavfile.write(str(filepath), sample_rate, (audio_loop * 32767).astype(np.int16))
        print(f"  Sauvegarde : {filepath.name}")

### Interpretation : Synthese programmatique

| Pattern | Observation |
|---------|------------|
| Rock basique | Pattern 4/4 standard, previsible |
| Funk syncope | Kick decale, groove plus complexe |
| Demi-temps | Espace, plus lent, atmospherique |

**Points cles** :
1. La synthese numpy est simple mais limitee en qualite sonore
2. Le format "positions en 16th notes" est intuitif pour decrire des patterns
3. Cette approche permet de generer des WAV sans aucune dependance externe
4. Un LLM peut generer des patterns dans ce format

## Section 5 : LLM genere des patterns programmatiques

On peut aussi utiliser le LLM pour generer des patterns dans notre format Python, en complement de Strudel.

In [ ]:
# LLM genere des patterns Python
print("LLM GENERE DES PATTERNS PYTHON")
print("=" * 45)

PATTERN_SYSTEM_PROMPT = """You generate drum patterns as Python data structures.
Format: list of (instrument, positions) tuples.
Instruments: "kick", "snare", "hihat"
Positions: list of integers 0-15 (16th notes in one 4/4 bar).
0=beat 1, 4=beat 2, 8=beat 3, 12=beat 4.

Return ONLY a Python list, no explanation.

Examples:
- "basic rock" -> [("kick", [0, 8]), ("snare", [4, 12]), ("hihat", [0, 2, 4, 6, 8, 10, 12, 14])]
- "four on the floor" -> [("kick", [0, 4, 8, 12]), ("snare", [4, 12]), ("hihat", [0, 2, 4, 6, 8, 10, 12, 14])]"""


def generate_pattern(description: str) -> Optional[list]:
    """Genere un pattern de batterie via LLM."""
    if not openai_available:
        return None

    try:
        response = client.chat.completions.create(
            model=llm_model,
            messages=[
                {"role": "system", "content": PATTERN_SYSTEM_PROMPT},
                {"role": "user", "content": description}
            ],
            temperature=0.5,
            max_tokens=200,
        )
        code = response.choices[0].message.content.strip()
        if code.startswith("```"):
            lines = code.split("\n")
            code = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])
        # Evaluer de maniere securisee
        import ast
        pattern = ast.literal_eval(code)
        return pattern
    except Exception as e:
        print(f"  Erreur : {type(e).__name__} - {str(e)[:100]}")
        return None


pattern_descriptions = [
    "a reggaeton beat with offbeat kick",
    "a jazz shuffle pattern",
    "a breakbeat with fast snare rolls",
]

if use_llm and openai_available:
    for desc in pattern_descriptions:
        print(f"\n--- {desc} ---")
        pattern = generate_pattern(desc)
        if pattern:
            print(f"  Pattern : {pattern}")
            audio = make_beat(pattern, bpm=bpm, sr=sample_rate)
            audio_loop = np.tile(audio, 4)
            display(Audio(data=audio_loop, rate=sample_rate))

            if save_results:
                safe_name = desc.lower().replace(" ", "_")[:30]
                filepath = OUTPUT_DIR / f"llm_{safe_name}.wav"
                wavfile.write(str(filepath), sample_rate, (audio_loop * 32767).astype(np.int16))
else:
    print("LLM desactive ou API non disponible")
    print("\nExemple de pattern genere par LLM :")
    example = [("kick", [0, 3, 8, 11]), ("snare", [4, 12]), ("hihat", list(range(16)))]
    print(f"  {example}")
    audio = make_beat(example, bpm=bpm, sr=sample_rate)
    display(Audio(data=np.tile(audio, 4), rate=sample_rate))

### Interpretation : Patterns generes par LLM

| Aspect | Observation |
|--------|------------|
| Fiabilite | Le format simple (listes d'entiers) est bien maitrise par les LLM |
| Securite | `ast.literal_eval` empeche l'execution de code arbitraire |
| Creativite | Les patterns sont musicalement coherents |
| Limites | Seulement 3 instruments dans cette implementation |

**Points cles** :
1. Le format structuré est plus sûr que de faire `exec()` sur du code LLM
2. On peut ajouter des instruments (clap, tom, cymbal) en etendant le dictionnaire
3. La combinaison LLM + synthese numpy donne un pipeline 100% autonome

## Section 6 : Iteration conversationnelle

La vraie puissance du LLM-driven live coding est l'**iteration conversationnelle** : on genere, on ecoute, on raffine.

| Etape | Action | Exemple |
|-------|--------|--------|
| 1 | Description initiale | "Un beat hip-hop" |
| 2 | Ecoute + feedback | "Plus de swing, kick plus lourd" |
| 3 | Raffinement | Code modifie |
| 4 | Iteration | "Ajoute un clap sur le 2 et le 4" |

In [9]:
# Pipeline d'iteration conversationnelle
print("ITERATION CONVERSATIONNELLE")
print("=" * 45)


class MusicSession:
    """Session de composition musicale avec historique."""

    def __init__(self, llm_model, system_prompt):
        self.messages = [{"role": "system", "content": system_prompt}]
        self.llm_model = llm_model
        self.history = []

    def compose(self, instruction: str) -> Optional[str]:
        """Envoie une instruction et recoit du code."""
        if not openai_available:
            return None

        self.messages.append({"role": "user", "content": instruction})

        try:
            response = client.chat.completions.create(
                model=self.llm_model,
                messages=self.messages,
                temperature=0.7,
                max_tokens=500,
            )
            code = response.choices[0].message.content.strip()
            if code.startswith("```"):
                lines = code.split("\n")
                code = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

            self.messages.append({"role": "assistant", "content": code})
            self.history.append({"instruction": instruction, "code": code})
            return code
        except Exception as e:
            print(f"  Erreur : {type(e).__name__} - {str(e)[:100]}")
            return None


# Demonstration d'iteration
if use_llm and openai_available:
    session = MusicSession(llm_model, STRUDEL_SYSTEM_PROMPT)

    iterations = [
        "Create a lo-fi hip hop beat with a lazy kick and soft hi-hats",
        "Add a mellow piano melody on top",
        "Make it more dreamy with reverb and a vinyl crackle effect",
    ]

    for i, instruction in enumerate(iterations):
        print(f"\n--- Iteration {i+1} ---")
        print(f"  Instruction : {instruction}")
        code = session.compose(instruction)
        if code:
            print(f"  Code : {code[:120]}")
            print(f"  REPL : {strudel_link(code)[:60]}...")

    # Afficher l'historique
    if session.history:
        print(f"\n\nHISTORIQUE DE SESSION")
        print(f"{'#':<4} {'Instruction':<50} {'Code (debut)':<50}")
        print("-" * 104)
        for i, entry in enumerate(session.history):
            print(f"{i+1:<4} {entry['instruction'][:50]:<50} {entry['code'][:50]:<50}")
else:
    print("LLM desactive ou API non disponible")
    print("\nExemple d'iteration :")
    print("  1. 'lo-fi hip hop beat' -> s(\"bd ~ bd ~, ~ sd ~ sd\").bank(\"RolandTR808\")")
    print("  2. 'add piano melody' -> stack(s(\"bd ...\"), note(\"c3 e3 g3 b3\").s(\"piano\"))")
    print("  3. 'more dreamy' -> ...room(0.8).delay(0.3).lpf(600)")

ITERATION CONVERSATIONNELLE
LLM desactive ou API non disponible

Exemple d'iteration :
  1. 'lo-fi hip hop beat' -> s("bd ~ bd ~, ~ sd ~ sd").bank("RolandTR808")
  2. 'add piano melody' -> stack(s("bd ..."), note("c3 e3 g3 b3").s("piano"))
  3. 'more dreamy' -> ...room(0.8).delay(0.3).lpf(600)


### Interpretation : Iteration conversationnelle

| Iteration | Observation |
|-----------|------------|
| 1 (initial) | Pattern de base coherent avec la description |
| 2 (ajout) | Le LLM conserve le contexte et ajoute par-dessus |
| 3 (raffinement) | Ajout d'effets sans casser le pattern existant |

**Points cles** :
1. L'historique conversationnel est essentiel pour le raffinement
2. Le LLM comprend les instructions de modification incrementale
3. 3-5 iterations suffisent generalement pour obtenir un resultat satisfaisant
4. Le code final peut etre sauvegarde et reutilise dans Strudel

In [10]:
# Mode interactif
if notebook_mode == "interactive" and not skip_widgets:
    print("MODE INTERACTIF - COMPOSEZ AVEC LE LLM")
    print("=" * 55)
    print("\nDecrivez la musique souhaitee en langage naturel :")
    print("  Exemples : 'drum and bass beat', 'calm ambient pad', 'jazz swing'")
    print("  (Laissez vide pour passer)")

    try:
        user_desc = input("\nDescription : ").strip()
        if user_desc and openai_available:
            code = generate_strudel(user_desc)
            if code:
                print(f"\nCode genere : {code}")
                print(f"REPL : {strudel_link(code)}")
                display(strudel_editor(code))
        elif user_desc:
            print("API OpenAI non disponible")
        else:
            print("Mode interactif ignore")

    except (KeyboardInterrupt, EOFError):
        print("Mode interactif interrompu")
    except Exception as e:
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("Mode interactif non disponible (execution automatisee)")
        else:
            print(f"Erreur : {error_type} - {str(e)[:100]}")
else:
    print("Mode batch - Interface interactive desactivee")

MODE INTERACTIF - COMPOSEZ AVEC LE LLM

Decrivez la musique souhaitee en langage naturel :
  Exemples : 'drum and bass beat', 'calm ambient pad', 'jazz swing'
  (Laissez vide pour passer)
Mode interactif non disponible (execution automatisee)


## Bonnes pratiques

### Prompt engineering musical

| Element | Bon exemple | Mauvais exemple |
|---------|------------|----------------|
| Genre | "lo-fi hip hop" | "musique" |
| Rythme | "syncopated, swing" | "interessant" |
| Instruments | "deep kick, soft hi-hats" | "des sons" |
| Effets | "reverb, vinyl crackle" | "sympa" |

### Securite du code genere

| Approche | Risque | Recommandation |
|----------|--------|----------------|
| `exec()` sur code LLM | Eleve | Eviter en production |
| `ast.literal_eval()` | Faible | Preferer pour les donnees |
| Strudel embed | Nul | Code JS sandbox dans le navigateur |
| Formats structures | Nul | Patterns comme listes Python |

In [11]:
# Statistiques de session
print("STATISTIQUES DE SESSION")
print("=" * 45)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"LLM : {llm_model if openai_available else 'non disponible'}")
print(f"BPM : {bpm}")
print(f"Codes Strudel generes : {len(generated_codes)}")
print(f"Beats numpy generes : {len(beat_results)}")

if save_results:
    wav_files = list(OUTPUT_DIR.glob('*.wav'))
    total_size = sum(f.stat().st_size for f in wav_files) / (1024*1024)
    print(f"\nFichiers WAV : {len(wav_files)} ({total_size:.1f} MB)")
    print(f"Repertoire : {OUTPUT_DIR}")

print(f"\nPROCHAINES ETAPES")
print(f"1. Explorer Strudel.cc directement dans le navigateur")
print(f"2. Combiner avec MusicGen (02-3) pour l'accompagnement")
print(f"3. Ajouter des samples MIDI (02-6) au pipeline")
print(f"4. Creer un bot Discord/Slack qui genere de la musique")

print(f"\nNotebook Live Coding Musical termine - {datetime.now().strftime('%H:%M:%S')}")

STATISTIQUES DE SESSION
Date : 2026-03-22 13:36:15
LLM : non disponible
BPM : 120
Codes Strudel generes : 3
Beats numpy generes : 3

Fichiers WAV : 3 (2.0 MB)
Repertoire : MyIA.AI.Notebooks\GenAI\outputs\audio\livecoding

PROCHAINES ETAPES
1. Explorer Strudel.cc directement dans le navigateur
2. Combiner avec MusicGen (02-3) pour l'accompagnement
3. Ajouter des samples MIDI (02-6) au pipeline
4. Creer un bot Discord/Slack qui genere de la musique

Notebook Live Coding Musical termine - 13:36:15


---

# EXERCICE - DJ IA

## Objectif

Utiliser le LLM pour creer une mini-performance musicale en 3 morceaux avec transitions.

## Consignes

1. **Choisir un theme** parmi :
   - "Du matin au soir" (calme -> energique -> calme)
   - "Tour du monde" (3 genres differents)
   - "Evolution" (simple -> complexe -> minimal)

2. **Generer 3 morceaux** via le LLM :
   - Morceau 1 : description + code Strudel
   - Morceau 2 : description + code Strudel
   - Morceau 3 : description + code Strudel

3. **Creer les transitions** :
   - Utiliser l'iteration conversationnelle pour modifier progressivement
   - Documenter les instructions donnees au LLM

4. **Evaluer** :
   - Coherence de chaque morceau (1-5)
   - Fluidite des transitions (1-5)
   - Respect du theme (1-5)

## Indices

- Utilisez `MusicSession` pour conserver le contexte entre les morceaux
- Strudel supporte le crossfade via `.gain()` progressif
- Commencez par des descriptions simples, raffinez ensuite

## Criteres de succes

- [ ] 3 morceaux generes avec codes Strudel valides
- [ ] Instructions LLM documentees pour chaque etape
- [ ] Evaluation comparative avec justification
- [ ] Liens REPL fonctionnels ou fichiers WAV

---

**Soumission** : PR avec titre "Exercice DJ IA - [Votre Nom]", codes, instructions et evaluation